In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as models
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

In [2]:
# --- GLOBAL CONFIG ---
PERCENTILE = 99.0
CROP_MARGIN = 2
CALIBRATION_BATCHES = 20
MAX_BATCHES = 100
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Workspace initialized. Using device: {device}')

torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Workspace initialized. Using device: cpu


In [3]:
def make_snn_ready(module):
    """
    Recursively replace ReLU / ReLU6 with physically instantiated
    nn.ReLU(inplace=False) so forward hooks can read clean voltages.
    """
    for child_name, child in module.named_children():
        if isinstance(child, (nn.ReLU, nn.ReLU6)):
            setattr(module, child_name, nn.ReLU(inplace=False))
        else:
            make_snn_ready(child)

In [4]:
print('--- Procuring Baseline Architecture ---')
continuous_model = models.mobilenet_v2(weights='DEFAULT')
make_snn_ready(continuous_model)
continuous_model.eval()
continuous_model = continuous_model.to(device)
print('Continuous backbone procured, refactored, and locked.')

--- Procuring Baseline Architecture ---
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\sagni/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:02<00:00, 6.50MB/s]


Continuous backbone procured, refactored, and locked.


In [5]:
class SimpleDepthDecoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.encoder = backbone.features
        self.decoder = nn.Sequential(
            nn.Conv2d(1280, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=False),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=False),
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=False),
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False),
            nn.Conv2d(64, 1, kernel_size=3, padding=1),
            nn.ReLU(inplace=False),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def calculate_rmse(pred, target):
    mse = F.mse_loss(pred, target)
    return torch.sqrt(mse).item()

In [6]:
# =============================================================================
# DATASET LOADING (PLACEHOLDER)
# Populate this cell with the TartanAir / KITTI dataloaders later.
#
# Required contract:
#   images:      [B, 3, 224, 224]   ImageNet-normalized RGB
#   gt_depths:   [B, 1, 224, 224]   depth in meters (aligned crop/scale to image)
#
#   val_loader    -> used for calibration + validation
#   train_loader  -> used for the decoder training loop
# =============================================================================
val_loader = None
train_loader = None

In [7]:
def run_validation(model, loader, limit=MAX_BATCHES, desc='Validating'):
    total_rmse = 0.0
    n = 0
    with torch.no_grad():
        for images, gt_depths in tqdm(loader, total=limit, desc=desc):
            images = images.to(device)
            gt_depths = gt_depths.to(device)
            total_rmse += calculate_rmse(model(images), gt_depths)
            n += 1
            if n >= limit:
                break
    return total_rmse / n


depth_model = SimpleDepthDecoder(continuous_model).to(device)
depth_model.eval()

print('--- Phase 1: Continuous Golden Baseline ---')
golden_rmse = run_validation(depth_model, val_loader, desc='Calculating Golden RMSE')
print(f'Continuous Golden Baseline RMSE: {golden_rmse:.4f}')

--- Phase 1: Continuous Golden Baseline ---


Calculating Golden RMSE:   0%|          | 0/100 [00:00<?, ?it/s]

TypeError: 'NoneType' object is not iterable

In [8]:
# --- Spatial-Masked Channel-Wise Calibration Engine ---
spatial_profiles = {}


def get_spatial_profile(layer_name):
    def hook(model, input, output):
        if layer_name not in spatial_profiles:
            spatial_profiles[layer_name] = []
        spatial_profiles[layer_name].append(output.detach().cpu())
    return hook


hooks = []
for name, module in continuous_model.named_modules():
    if isinstance(module, nn.ReLU):
        hooks.append(module.register_forward_hook(get_spatial_profile(name)))

print('Running Spatial Calibration (multi-batch)...')
with torch.no_grad():
    for i, (images, _) in enumerate(tqdm(val_loader, total=CALIBRATION_BATCHES, desc='Profiling Voltages')):
        _ = continuous_model(images.to(device))
        if i >= CALIBRATION_BATCHES - 1:
            break

for handle in hooks:
    handle.remove()
print(f'Calibration complete. Profiled {len(spatial_profiles)} layers.')

Running Spatial Calibration (multi-batch)...


Profiling Voltages:   0%|          | 0/20 [00:00<?, ?it/s]

TypeError: 'NoneType' object is not iterable

In [9]:
def compute_channel_thresholds(layer_name, percentile=PERCENTILE, crop_margin=CROP_MARGIN):
    stacked = torch.cat(spatial_profiles[layer_name], dim=0)  # [B_total, C, H, W]
    C, H, W = stacked.shape[1], stacked.shape[2], stacked.shape[3]
    theta = np.zeros(C)
    for c in range(C):
        channel_map = stacked[:, c, :, :]
        if H > crop_margin * 2 and W > crop_margin * 2:
            safe_core = channel_map[:, crop_margin:-crop_margin, crop_margin:-crop_margin]
        else:
            safe_core = channel_map
        theta[c] = np.percentile(safe_core.numpy(), percentile)
    return theta


channel_thresholds = {}
for layer_name in spatial_profiles:
    channel_thresholds[layer_name] = compute_channel_thresholds(layer_name)

first_layer = list(channel_thresholds.keys())[0]
print(f'Computed channel-wise thresholds for {len(channel_thresholds)} layers.')
print(f'Sample threshold count for {first_layer}: {len(channel_thresholds[first_layer])}')

IndexError: list index out of range

In [10]:
class StrictT1SFN(nn.Module):
    def __init__(self, thresholds):
        super().__init__()
        self.register_buffer('thresholds', torch.tensor(thresholds, dtype=torch.float32).view(1, -1, 1, 1))

    def forward(self, x):
        spikes = (x >= self.thresholds).float()
        return spikes * self.thresholds


def convert_to_snn(module, prefix=''):
    for name, child in module.named_children():
        full_name = f'{prefix}.{name}' if prefix else name
        if isinstance(child, nn.ReLU):
            if full_name in channel_thresholds:
                setattr(module, name, StrictT1SFN(thresholds=channel_thresholds[full_name]).to(device))
        else:
            convert_to_snn(child, full_name)


print('--- Phase 2: SNN Conversion Surgery ---')
convert_to_snn(continuous_model)
print('Surgery complete. Backbone is now a strict T=1 Scale-and-Fire SNN.')

--- Phase 2: SNN Conversion Surgery ---
Surgery complete. Backbone is now a strict T=1 Scale-and-Fire SNN.


In [11]:
# --- Feature-Level Fidelity Check ---
golden_encoder = models.mobilenet_v2(weights='DEFAULT').features.to(device)
golden_encoder.eval()

sample_images, _ = next(iter(val_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    continuous_features = golden_encoder(sample_images)
    spiking_features = continuous_model.features(sample_images)

feature_mse = F.mse_loss(spiking_features, continuous_features).item()
print(f'Feature-Level MSE (Continuous vs. SNN): {feature_mse:.6f}')

cont_map = continuous_features[0].mean(dim=0).cpu().numpy()
snn_map = spiking_features[0].mean(dim=0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cont_map, cmap='magma')
axes[0].set_title('Continuous Feature Map')
axes[0].axis('off')
axes[1].imshow(snn_map, cmap='magma')
axes[1].set_title(f'Spiking Feature Map (T=1 SFN)\nMSE: {feature_mse:.4f}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

TypeError: 'NoneType' object is not iterable

In [12]:
# --- Phase 3: Untrained Decoder Baseline ---
final_depth_model = SimpleDepthDecoder(continuous_model).to(device)
final_depth_model.eval()

snn_rmse_untrained = run_validation(final_depth_model, val_loader, desc='Calculating Untrained SNN RMSE')
print(f'Untrained T=1 SNN RMSE (random decoder): {snn_rmse_untrained:.4f}')

Calculating Untrained SNN RMSE:   0%|          | 0/100 [00:00<?, ?it/s]

TypeError: 'NoneType' object is not iterable

In [13]:
# --- Phase 4: End-to-End Decoder Training ---
def compute_depth_loss(predicted, target, alpha=0.1):
    rmse = torch.sqrt(F.mse_loss(predicted, target))
    grad_x = torch.mean(torch.abs(predicted[:, :, :, :-1] - predicted[:, :, :, 1:]))
    grad_y = torch.mean(torch.abs(predicted[:, :, :-1, :] - predicted[:, :, 1:, :]))
    return rmse + alpha * (grad_x + grad_y)


snn_backbone = continuous_model
for param in snn_backbone.parameters():
    param.requires_grad = False
snn_backbone.eval()

for param in final_depth_model.decoder.parameters():
    assert param.requires_grad, 'Decoder gradient is frozen!'

optimizer = torch.optim.AdamW(final_depth_model.decoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

final_depth_model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0
    for images, gt_depths in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{NUM_EPOCHS}'):
        images = images.to(device)
        gt_depths = gt_depths.to(device)

        optimizer.zero_grad()

        with torch.no_grad():
            spiking_features = snn_backbone.features(images)

        predicted_depths = final_depth_model.decoder(spiking_features)
        loss = compute_depth_loss(predicted_depths, gt_depths)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    print(f'Epoch {epoch + 1}: avg train loss = {epoch_loss / n_batches:.4f}')

final_depth_model.eval()
trained_rmse = run_validation(final_depth_model, val_loader, desc='Calculating Trained SNN RMSE')
print(f'Trained T=1 SNN RMSE: {trained_rmse:.4f}')

Epoch 1/10: 0it [00:00, ?it/s]

TypeError: 'NoneType' object is not iterable

In [ ]:
print('=========================================')
print('          FINAL SYSTEM REPORT            ')
print('=========================================')
print(f'Continuous Golden Baseline RMSE:  {golden_rmse:.4f}')
print(f'Untrained T=1 SNN RMSE:           {snn_rmse_untrained:.4f}')
print(f'Trained T=1 SNN RMSE:             {trained_rmse:.4f}')
print(f'Feature-Level MSE (Continuous vs. SNN): {feature_mse:.6f}')
print('=========================================')

torch.save(final_depth_model.state_dict(), 'trained_t1_snn_depth.pth')
print('Checkpoint saved: trained_t1_snn_depth.pth')

In [ ]:
# --- Visual Verification ---
with torch.no_grad():
    sample_images, sample_depths = next(iter(val_loader))
    sample_images = sample_images.to(device)
    sample_depths = sample_depths.to(device)
    pred_depths = final_depth_model(sample_images)

idx = 0
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.clip(sample_depths[idx, 0].cpu().numpy(), 0, 50), cmap='magma')
axes[0].set_title('Ground Truth Depth (clipped to 50m)')
axes[0].axis('off')
axes[1].imshow(np.clip(pred_depths[idx, 0].cpu().numpy(), 0, 50), cmap='magma')
axes[1].set_title('Predicted Depth')
axes[1].axis('off')
plt.tight_layout()
plt.show()